# Controlled omega_z super-resolution comparison (32 x 32 -> 128 x 128)

On PACE, open this notebook in a GPU allocation with the project `jhtdb-ml` kernel (see `environment.yml`). Edit only the settings cell below, then **Run All**. Launch Jupyter from the repository or one of its subdirectories; set `PROJECT_ROOT` explicitly if necessary. No extra packages beyond the project environment are required.

The default compares the corrected U-Net and residual CNN against one shared bicubic baseline. Add another compatible model config to `CONFIG_PATHS` for future comparisons, or select just one config. Models are built through the repository model factory.

**Freeze configs, trained runs, checkpoints, metric definitions, and example offsets before testing.** The supplied validation losses (U-Net 0.3548; residual CNN 0.3564) are context only. Every network loads the exact run's `checkpoints/best_model.pth`, selected during training by validation loss. There is no last-epoch fallback, run search, test-based checkpoint selection, or test leaderboard. Do not use this notebook's results to choose models, epochs, hyperparameters, or examples; use validation data for those decisions.


In [ ]:
from pathlib import Path

# ----- Edit this cell, then Run All -----
PROJECT_ROOT = None  # e.g. Path('/path/on/pace/jhtdb-deep-learning-vip')
CONFIG_PATHS = [
    'configs/vip_unet_corrected_physics_128.yaml',
    'configs/vip_residual_cnn_corrected_physics_128.yaml',
]
# Optional exact run-directory overrides, keyed by experiment.run_name.
# Useful when PACE checkpoints live on scratch. Absolute paths are supported.
RUN_DIRS = {}  # {'vip_unet_corrected_physics_x4': '/scratch/.../exact_run'}
DEVICE = 'cuda'  # Explicitly set 'cpu' only for a CPU evaluation.
WARMUP_FORWARDS = 20
EXAMPLE_TEST_OFFSETS = (0, 199, 399)  # Preselected, never best/worst test examples.
OUTPUT_BASE = 'work/SR/comparisons'  # Relative to project root, or absolute scratch path.


## Resolve configs and validation-selected checkpoints

The preflight checks require identical data, domain, and composite loss settings for this controlled study. It also checks each run's saved `config.yaml` when available; the JSON records whether that check was possible. A missing checkpoint stops execution. Configs are recorded in full and checkpoint SHA-256 hashes identify the weights evaluated. No training files are changed.


In [ ]:
import sys
from datetime import datetime, timezone

if PROJECT_ROOT is None:
    ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / 'models/factory.py').is_file() and (p / 'configs').is_dir()), None)
    if ROOT is None:
        raise FileNotFoundError('Set PROJECT_ROOT to the repository location on PACE.')
else:
    ROOT = Path(PROJECT_ROOT).expanduser().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display, Image
from utils.sr_comparison import prepare_runs, load_data, evaluate, save_results

runs = prepare_runs(ROOT, CONFIG_PATHS, RUN_DIRS)
for run in runs:
    print(run['name'], '\n ', run['checkpoint'])
    print('  Saved training config verified:', run['saved_config_verified'])


## Reproduce training preprocessing and the full held-out split

Snapshots use the training notebook's lexicographically sorted HDF5 keys. Indices 0-2999 train, 3000-3599 validate, and **3600-3999 test**. Global means/stds and the fixed PSNR/SSIM data range come only from training snapshots. Each field is normalized separately exactly as in `MultiFieldDataset`; LR is average-pooled by four before normalization. No augmentation, temporal stacking, shuffling, subsampling, or dropped batches is used.

MSE and MAE are reported in denormalized omega_z units. PSNR and repository SSIM share the same training max-minus-min range for every model and sample; predictions never determine it. SSIM uses the repository's Gaussian 11x11 window and zero-padding convention. Summary values are sample means and population standard deviations; PSNR is the mean of per-sample PSNRs.

`advection_error` and `diffusion_error` are **unweighted MSE terms matching the corrected training loss**: normalized fields, index spacing dx=dy=1, DNS velocities for both predictions and targets, central stencils on the 126x126 interior, and the repository's term clipping. Additional `physical_advection_mse` and `physical_diffusion_mse` use denormalized fields and configured physical grid spacings with no clipping. Diffusion here means Laplacian agreement without a viscosity factor. These are spatial operator errors, not a full time-dependent Navier-Stokes residual. The last spatial axis is treated as x, consistent with training.


In [ ]:
dataset, metadata = load_data(ROOT, runs[0]['config'])
assert len(dataset) == 400
print('Test samples:', len(dataset))
print('Snapshot indices:', metadata['test_indices'][0], 'through', metadata['test_indices'][-1])
print('Training-only normalization:', metadata['normalization_stats'])
print('Fixed training data range:', metadata['data_range'])


## Evaluate all selected models and bicubic

All methods use FP32, evaluation mode and inference mode. Timing uses batch size one with the same device and warmup count. CUDA is synchronized around each forward. Timing excludes transfers, disk reads, preprocessing, and metric computation, and includes synchronization overhead. The summary reports mean, standard deviation, and median milliseconds per sample. This measures forward latency, not end-to-end throughput; compare timings from the same PACE GPU allocation. CUDA must be available when selected; there is no silent CPU fallback.


In [ ]:
rows, summary, examples = evaluate(
    runs, dataset, metadata, device=DEVICE, warmup=WARMUP_FORWARDS,
    example_indices=EXAMPLE_TEST_OFFSETS,
)
assert len(rows) == 400 * (len(runs) + 1)
# Preserve configured order. Test scores are reporting only; never used to select or rank models.
summary_table = pd.DataFrame(summary).set_index('model')
display(summary_table[[
    'num_test_samples', 'mse', 'mae', 'psnr', 'ssim',
    'advection_error', 'diffusion_error', 'parameter_count',
    'inference_ms', 'inference_ms_std', 'inference_ms_median',
]])
display(summary_table[['physical_advection_mse', 'physical_diffusion_mse']])


## Save tables, provenance, and comparison plots

Each execution creates a new UTC-stamped directory so earlier comparisons remain intact. Outputs: `results.json` (aggregate metrics, exact configs, checkpoint hashes, split IDs/keys, normalization, definitions, device/precision and software metadata), `summary.csv`, `per_sample_metrics.csv`, metric and per-snapshot comparison plots, and field/error panels for the preselected offsets. Field panels share color scales and error panels share error scales across methods; no interpolation smoothing is applied.


In [ ]:
output_dir = (ROOT / OUTPUT_BASE) / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
save_results(output_dir, runs, metadata, rows, summary, examples, DEVICE, WARMUP_FORWARDS)
print('Results saved to:', output_dir)
display(Image(filename=str(output_dir / 'metric_comparison.png')))
display(Image(filename=str(output_dir / 'per_sample_comparison.png')))
for offset in EXAMPLE_TEST_OFFSETS:
    display(Image(filename=str(output_dir / f'fields_test_{offset:03d}.png')))
